# L4 · EfficientSAM3 vs SAM 3: a field-sized building finder? (GPU T4 + CPU)

Task **L4** from `Essential_task_delegation/Next_Steps_Roadmap.md` (it was step S5 in `Research_Directions_Beyond_Finetuning.md` §7.4). Goal **O4** (edge/field models).

**Run 1 (18 Nov) stopped in Cell 1. Fixed before run 2, with no result seen:**
- **Cell 1:** `pip install -e` registers the package through a hook that an already-running Kaggle kernel never reads, so `import sam3` failed. The clone's package folder now goes on `sys.path`, and the cell checks that `sam3` comes from the clone.
- **Cell 2:** run 1 guessed the students from file names. The guess misread hyphenated names (`efficientvit-b0` as `b2`), could pick a SAM 3.1 or a stage-1 checkpoint, and kept the text encoder at 77 tokens, which the released checkpoints (16) don't fit. The students are now the repo's three released full models, with its README's settings.
- **Cell 5:** each checkpoint must fill at least 95% of its model's parameters, because the repo's loader only prints missing keys. E3's `addmm_act` patch is dropped: this copy of SAM 3 has no such function, and the students don't use SAM 3's ViT.
- **Cell 7:** the rule's memory condition is now applied too.
- **The expectation** below now gives the three students' sizes; nothing else in it changed.

**Why**
- In **E3**, full SAM 3 (`facebook/sam3`, 841M parameters, 3.45 GB of weights) prompted with `"building"` on the before-photo in four 512 px quarters (`pre_grid2`) scored **Building IoU 0.6304** on all 645 validation pairs (**0.6302** on the 605 clean pairs) and covered **63.4%** of destroyed-building footprints (**86%** with R9's pieces), beating R9 with TTA (`0.5905`).
- But full SAM 3 took **2.11 s per image** (`pre_grid2`, 4.07 GB peak GPU memory) on a T4 and is server-sized.
- **EfficientSAM3** (Zeng et al., arXiv:2511.15833; repo `SimonZeng7108/efficientsam3`, weights in `Simon7108528/EfficientSAM3`) distils SAM 3 into small vision backbones (EfficientViT, RepViT, TinyViT) and MobileCLIP text encoders. Its paper reports no quantitative numbers yet ("Coming Soon"), so we measure it ourselves on our 645 validation pairs (and 605 clean pairs).

**What it does**
1. **Loads EfficientSAM3's three released full models** from `Simon7108528/EfficientSAM3` (the repo README's "Model Zoo"): **EV-M** (EfficientViT-B1 vision backbone, 89.2M parameters in all), **RV-M** (RepViT-M1.1, 92.7M) and **TV-M** (TinyViT-11M, 95.3M). Each has the MobileCLIP-S0 text encoder at a 16-token context, so it takes the text prompt `"building"`. They are built exactly as the README's "Quick Start" builds TV-M.
2. **Part 1 (T4 precision & CPU latency benchmark):**
   - On the densest validation image (`turkey_earthquake0_post_456.png`, 741,855 building pixels, the same probe as E3), runs each student on the T4 in `fp32` and `fp16`, recording `ms_per_image`, `peak_gpu_gb`, instance count, and `iou_vs_fp32`.
   - On the first 20 clean validation pairs (`LATENCY_CPU_PAIRS`), measures 4-thread **CPU latency** (`ms_per_image_cpu`, median after warm-up) for `pre_grid1` (whole image) and `pre_grid2` (four quarters).
3. **Part 2 (Building finding on the 645 validation before-photos, scored on all 645 and clean 605):**
   - Runs each student with the prompt `"building"` on the before-photo as a whole image (`pre_grid1`) and as four 512 px quarters (`pre_grid2`), identical to E3.
   - Computes dataset-level **Building IoU**, instance-to-blob **count ratio**, **count MAE**, and **footprint recall** per true class (`Intact`, `Damaged`, `Destroyed`) on all 645 pairs and on the **605 clean pairs** (E5's 40 leaked pairs excluded), overall and by source (`xBD` vs `other`).
   - Compares directly against E3's saved SAM 3 per-image counts (embedded in Cell 4 and verified against E3's published `0.6304` / `0.6302`) and R9 with TTA (`0.5905` all / `0.5905` clean).

**The rule (fixed before the run)**
- A student passes as a field building finder for the find-then-grade chain if, on the **605 clean validation pairs** (`pre_grid2`):
  1. its **Building IoU is at least R9's (`0.5905`, Gate G3's bar)** and within `0.05` of full SAM 3 (`>= 0.5802`);
  2. it runs faster and uses less GPU memory than full SAM 3 (`2,110 ms/image` and `4.07 GB` on T4).
- **If a student passes:** it becomes the candidate field finder for find-then-grade.
- **If none passes:** full SAM 3 stays the base-station finder, and the 8-bit U-Net (`L1b` R9 / `L1c` R11, 33 MB) remains the field model, with the measured trade-off table reported for O4.

**Expectation (fixed before the run)**
- The students are about 9× smaller than full SAM 3 (89–95M parameters against 841M) and faster on T4 and CPU.
- Because EfficientSAM3's distillation trained on general concepts rather than overhead satellite imagery, zero-shot aerial `"building"` recall may drop substantially compared to full SAM 3 (`0.6302`), likely landing below R9's `0.5905`. Either outcome gives O4's measured trade-off table.

**How to run on Kaggle** (any free account with GPU T4 ×2)
1. Settings: Accelerator **GPU T4 x2** (one is used), Internet **On**. No secret is needed: `Simon7108528/EfficientSAM3` is public.
2. Input: `abrarmohammedtanzim/disasterm3-mirror`.
3. Run All once with `SMOKE_TEST = True` in Cell 2 (8 validation pairs, 3 CPU latency pairs; about 10–15 min). Then set `SMOKE_TEST = False` and **Save Version → Save & Run All (Commit)**. About 1.5–2 hours with three students (estimated).
4. Outputs in `/kaggle/working/L4/`: `smoke_and_latency.csv`, `l4_per_image.csv`, `L4_summary.csv`, `config.json`, `README.md`.

In [ ]:
# ── Cell 1: Install EfficientSAM3 (SimonZeng7108/efficientsam3) and dependencies ──
# Like SAM 3 in E3, EfficientSAM3's repo pins numpy<2 in pyproject.toml; installing with --no-deps keeps
# Kaggle's numpy 2.x and torch untouched while installing the packages it imports.
import importlib.util
import os
import subprocess
import sys
from pathlib import Path

ESAM3_REPO = "https://github.com/SimonZeng7108/efficientsam3.git"
ESAM3_DIR = Path("/tmp/efficientsam3")

def pip(*args):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *args], check=True)

if not ESAM3_DIR.exists():
    subprocess.run(["git", "clone", "--depth", "1", ESAM3_REPO, str(ESAM3_DIR)], check=True)
ESAM3_COMMIT = subprocess.check_output(["git", "-C", str(ESAM3_DIR), "rev-parse", "HEAD"], text=True).strip()

# Run 1 (18 Nov) failed here: `pip install -e` registers the package through a .pth finder that the already-running
# Kaggle kernel never reads, so `import sam3` raised ModuleNotFoundError. Instead, put the clone's folder that
# holds sam3/model_builder.py on sys.path directly (no pip install of the repo; its numpy<2 pin is avoided too).
_builders = sorted(ESAM3_DIR.rglob("sam3/model_builder.py"), key=lambda p: len(p.parts))
assert _builders, f"sam3/model_builder.py not found in the clone; top level: {sorted(p.name for p in ESAM3_DIR.iterdir())}"
ESAM3_PKG_ROOT = _builders[0].parent.parent
if str(ESAM3_PKG_ROOT) not in sys.path:
    sys.path.insert(0, str(ESAM3_PKG_ROOT))
for _m in [m for m in sys.modules if m == "sam3" or m.startswith("sam3.")]:
    del sys.modules[_m]          # drop any stale sam3 from an earlier attempt in this session
importlib.invalidate_caches()
print(f"sam3 package found at {ESAM3_PKG_ROOT / 'sam3'}")
pip("timm>=1.0.17", "ftfy==6.1.1", "regex", "iopath", "einops", "pycocotools", "psutil",
    "open_clip_torch", "huggingface_hub", "safetensors")
if importlib.util.find_spec("pkg_resources") is None:
    pip("setuptools<81")

try:
    from kaggle_secrets import UserSecretsClient
    tok = UserSecretsClient().get_secret("HF_TOKEN")
    if tok:
        os.environ["HF_TOKEN"] = tok
except Exception:
    pass

import torch
import sam3
import sam3.model_builder as mb
assert Path(sam3.__file__).resolve().is_relative_to(ESAM3_DIR.resolve()), f"wrong sam3 imported: {sam3.__file__}"
assert hasattr(mb, "build_efficientsam3_image_model"), "build_efficientsam3_image_model not found in sam3.model_builder"
print(f"✓ EfficientSAM3 installed (commit {ESAM3_COMMIT[:7]}); torch {torch.__version__}, python {sys.version.split()[0]}")
print(f"  GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none (needs GPU T4)'}")

In [ ]:
# ── Cell 2: Configuration, E5's 40 leaked pairs, and student discovery in Simon7108528/EfficientSAM3 ──
import inspect
import json
import re
import time
from contextlib import nullcontext
from pathlib import Path

import numpy as np
from huggingface_hub import HfApi, hf_hub_download

SMOKE_TEST = False                  # True = 8 validation pairs, 3 CPU latency pairs
DATA_ROOT = Path("/kaggle/input/datasets/abrarmohammedtanzim/disasterm3-mirror/DisasterM3_Instruct")
MANIFEST_PATH = DATA_ROOT / "train_release.json"
OUT_DIR = Path("/kaggle/working/L4")
OUT_DIR.mkdir(parents=True, exist_ok=True)

HF_REPO = "Simon7108528/EfficientSAM3"
PROMPT = "building"
CONFIDENCE = 0.5                    # same threshold as E3
RUNS = [("pre", 1), ("pre", 2)]     # pre_grid1 (1024 px whole) and pre_grid2 (2x2 quarters of 512 px, E3's winner)
MIN_INSTANCE_PX = 10                # ignore specks, as in E3
LATENCY_CPU_PAIRS = 3 if SMOKE_TEST else 20
CPU_THREADS = 4                     # same thread count as L1b and L1c
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# E5 step 1's 40 leaked validation pairs (val_train_leaks_hash.csv, counted True)
LEAK40 = {f"socal_fire_{n:08d}_post_disaster.png" for n in
          (81, 188, 218, 220, 222, 235, 301, 341, 348, 396, 436, 455, 497, 504, 614, 1188)} | \
         {f"woolsey_fire_{n:08d}_post_disaster.png" for n in
          (9, 30, 32, 137, 272, 399, 435, 474, 498, 514, 529, 573, 576, 589, 623, 632, 634, 653, 679, 703, 722, 758,
           783, 810)}
assert len(LEAK40) == 40
assert MANIFEST_PATH.exists(), "Attach abrarmohammedtanzim/disasterm3-mirror"

# The students: EfficientSAM3's three released full models (the repo README's "Model Zoo"), each a small vision
# backbone with the MobileCLIP-S0 text encoder at a 16-token context, built as the README's "Quick Start" builds TV-M.
# Run 1 guessed students from file names instead (see Cell 0).
TEXT_ENCODER, TEXT_CONTEXT = "MobileCLIP-S0", 16
STUDENTS = [
    {"label": "EV-M", "rfilename": "efficientsam3_ft/efficientsam3_efficientvit.pt",
     "backbone_type": "efficientvit", "model_name": "b1", "readme_params_m": 89.2},
    {"label": "RV-M", "rfilename": "efficientsam3_ft/efficientsam3_repvit.pt",
     "backbone_type": "repvit", "model_name": "m1.1", "readme_params_m": 92.7},
    {"label": "TV-M", "rfilename": "efficientsam3_ft/efficientsam3_tinyvit.pt",
     "backbone_type": "tinyvit", "model_name": "11m", "readme_params_m": 95.3},
]
api = HfApi(token=os.environ.get("HF_TOKEN"))
info = api.model_info(HF_REPO, files_metadata=True)
HF_SHA = info.sha
sizes = {s.rfilename: s.size for s in info.siblings}
for st in STUDENTS:
    assert st["rfilename"] in sizes, f"{st['rfilename']} isn't in {HF_REPO}@{HF_SHA[:8]}"
    st.update(size_bytes=sizes[st["rfilename"]], text_encoder_type=TEXT_ENCODER, text_context=TEXT_CONTEXT)
print(f"✓ {HF_REPO}@{HF_SHA[:8]}: EfficientSAM3's three released full models")
for st in STUDENTS:
    print(f"  - {st['label']}: {st['rfilename']} ({(st['size_bytes'] or 0) / 1e6:.0f} MB), {st['backbone_type']} "
          f"{st['model_name']}, {TEXT_ENCODER} at {TEXT_CONTEXT} tokens (README: {st['readme_params_m']}M parameters)")

In [ ]:
# ── Cell 3: Load manifest and build the 4-class masks (verbatim from E3 and the D4 notebooks) ──
import cv2
import numpy as np
import os
from collections import Counter, defaultdict
from pathlib import Path
import json

with open(MANIFEST_PATH, "r", encoding="utf-8") as f:
    raw_data = json.load(f)
print(f"Loaded {len(raw_data):,} entries from manifest")

bda_entries = [
    e for e in raw_data
    if e.get("task") == "Referring Expression Segmentation"
    and e.get("cls_description") == "Building Damage Assessment"
]
print(f"Building Damage Assessment entries: {len(bda_entries):,}")

def resolve_path(rel_path):
    if not rel_path:
        return None
    rel_path = rel_path.replace("\\", "/")
    filename = Path(rel_path).name
    candidates = [
        DATA_ROOT / rel_path,
        DATA_ROOT / "train_images" / rel_path,
        DATA_ROOT / "train_images" / "train_images" / filename,
        DATA_ROOT / "masks" / rel_path,
        DATA_ROOT / "masks" / "masks" / filename,
    ]
    for c in candidates:
        if c.exists():
            return str(c)
    return None

by_image = defaultdict(dict)
for e in bda_entries:
    post_img = e.get("post_image_path", "")
    if e.get("image_type") != "Optical":
        continue
    mask_rel = e.get("ground_truth", "")
    folder = Path(mask_rel.replace("\\", "/")).parent.name
    resolved = resolve_path(mask_rel)
    if resolved:
        by_image[post_img][folder] = resolved
        by_image[post_img]["post_image_path"] = post_img
        by_image[post_img]["pre_image_path"] = e.get("pre_image_path")

print(f"Unique images with at least one mask: {len(by_image):,}")

FOLDER_TO_CLASS = {
    "train_building_intact_mask": 1,
    "train_building_damaged_mask": 2,
    "train_building_destroyed_mask": 3,
}

os.makedirs("/tmp/combined_masks", exist_ok=True)
pairs = []
skipped_img = 0
for post_img, mask_dict in by_image.items():
    img_path = resolve_path(post_img)
    if not img_path:
        skipped_img += 1
        continue
    combined_mask = None
    for folder, class_id in FOLDER_TO_CLASS.items():
        mask_path = mask_dict.get(folder)
        if mask_path is None:
            continue
        m = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
        if m is None:
            continue
        if combined_mask is None:
            combined_mask = np.zeros_like(m, dtype=np.uint8)
        combined_mask[m > 0] = class_id
    if combined_mask is None:
        continue
    mask_filename = Path(img_path).name
    save_path = f"/tmp/combined_masks/{mask_filename}"
    cv2.imwrite(save_path, combined_mask)
    pairs.append({
        "image_path": img_path,
        "mask_path": save_path,
        "pre_path": resolve_path(mask_dict.get("pre_image_path")),
    })

assert len(pairs) == 6443, f"Expected 6,443 pairs, got {len(pairs)}"
print(f"\n✓ Built {len(pairs):,} valid (image, mask) pairs (skipped: {skipped_img})")

In [ ]:
# ── Cell 4: Validation split (645 pairs, 605 clean) and E3's embedded SAM 3 reference counts ──
import base64
import gzip
import io
import pandas as pd
from sklearn.model_selection import train_test_split

train_pairs, val_pairs = train_test_split(pairs, test_size=0.1, random_state=42)
assert len(val_pairs) == 645
assert all(p["pre_path"] is not None for p in val_pairs), "A validation pair is missing its before-image"

XBD_NAME = re.compile(r"^(.+?)_(\d{8})_post_disaster")
OTHER_NAME = re.compile(r"^(.+?)_post")

def event_and_source(image_path):
    name = Path(image_path).name
    m = XBD_NAME.match(name)
    if m:
        return m.group(1), "xBD"
    m = OTHER_NAME.match(name)
    return re.sub(r"\d+$", "", m.group(1) if m else Path(name).stem), "other"

for p in val_pairs:
    p["image"] = Path(p["image_path"]).name
    p["event"], p["source"] = event_and_source(p["image_path"])
    p["clean"] = p["image"] not in LEAK40

n_clean = sum(p["clean"] for p in val_pairs)
assert n_clean == 605, f"Expected 605 clean validation pairs, found {n_clean}"

# E3's per-image SAM 3 results (Champion_Task/E3-results/E3/sam3_s1.csv), embedded so L4 reproduces
# SAM 3's exact all-645 and clean-605 numbers on any Kaggle account without needing E3 attached.
E3_SAM3_B64 = "..."
E3_DF = pd.read_csv(io.BytesIO(gzip.decompress(base64.b64decode(E3_SAM3_B64))))
E3_DF["clean"] = ~E3_DF["image"].isin(LEAK40)

CLASSES = ((1, "intact"), (2, "damaged"), (3, "destroyed"))

def summarize_df(df):
    return {
        "images": len(df),
        "building_iou": round(float(df["inter"].sum() / max(df["union"].sum(), 1)), 4),
        "count_ratio": round(float(df["instances"].sum() / max(df["gt_blobs"].sum(), 1)), 3),
        "count_mae": round(float((df["instances"] - df["gt_blobs"]).abs().mean()), 1),
        **{f"recall_{c}": round(float(df[f"found_{c}_px"].sum() / max(df[f"gt_{c}_px"].sum(), 1)), 4) for _, c in CLASSES},
        "sec_per_image": round(float(df["sec"].mean()), 2) if "sec" in df and df["sec"].notna().any() else np.nan,
    }

# Self-check: E3_DF must reproduce E3's published numbers on all 645 and clean 605 (Task_Ledger E3)
_e3_all = summarize_df(E3_DF[E3_DF["run"] == "pre_grid2"])
_e3_clean = summarize_df(E3_DF[(E3_DF["run"] == "pre_grid2") & E3_DF["clean"]])
assert _e3_all["building_iou"] == 0.6304 and _e3_all["recall_destroyed"] == 0.6339, _e3_all
assert _e3_clean["building_iou"] == 0.6302, _e3_clean

# Probe image = the densest validation image, identical to E3 Part 1
gt_building_px = [int((cv2.imread(p["mask_path"], cv2.IMREAD_GRAYSCALE) > 0).sum()) for p in val_pairs]
PROBE_PAIR = val_pairs[int(np.argmax(gt_building_px))]
PROBE_MAX_PX = max(gt_building_px)
assert PROBE_PAIR["image"] == "turkey_earthquake0_post_456.png" and PROBE_MAX_PX == 741855

if SMOKE_TEST:
    val_pairs = val_pairs[:8]
print(f"✓ Validation pairs: {len(val_pairs)} (clean: {sum(p['clean'] for p in val_pairs)}); "
      f"E3 SAM 3 pre_grid2 verified: all 645 = {_e3_all['building_iou']}, clean 605 = {_e3_clean['building_iou']}")
print(f"  Probe image: {PROBE_PAIR['image']} ({PROBE_MAX_PX:,} ground-truth building pixels)")

In [ ]:
# ── Cell 5: Load students, Part 1 smoke test on T4 (fp32 vs fp16) and CPU latency benchmark ──
import gc
from PIL import Image
from tqdm.auto import tqdm
from sam3.model.sam3_image_processor import Sam3Processor
from sam3.model_builder import build_efficientsam3_image_model

# Run 1 copied E3's `vitdet.addmm_act` patch here. It is dropped: this copy of SAM 3 has no `addmm_act`, and the
# students' vision backbones aren't SAM 3's ViT.


def checkpoint_coverage(model, ckpt):
    """Share of the model's parameter values whose names the checkpoint holds, after the key cleaning that
    sam3.model_builder._load_checkpoint applies (it loads with strict=False and only prints missing keys)."""
    sd = torch.load(ckpt, map_location="cpu", weights_only=True)
    sd = sd["model"] if isinstance(sd.get("model"), dict) else sd

    def clean(k):
        if k.startswith("detector."):
            k = k.replace("detector.", "")
        return k.replace("student_trunk.", "")

    names = {clean(k) for k in sd}
    msd = model.state_dict()
    return sum(v.numel() for k, v in msd.items() if k in names) / max(sum(v.numel() for v in msd.values()), 1)


def load_student(st, device=DEVICE):
    """Downloads the checkpoint (pinned to HF_SHA) and builds the model as the repo README's Quick Start does."""
    ckpt = hf_hub_download(repo_id=HF_REPO, filename=st["rfilename"], revision=HF_SHA,
                           token=os.environ.get("HF_TOKEN"))
    model = build_efficientsam3_image_model(
        checkpoint_path=ckpt, backbone_type=st["backbone_type"], model_name=st["model_name"],
        text_encoder_type=st["text_encoder_type"], text_encoder_context_length=st["text_context"],
        load_from_HF=False, device=device)
    model.to(device).eval()
    if "coverage" not in st:
        st["coverage"] = round(checkpoint_coverage(model, ckpt), 4)
        assert st["coverage"] >= 0.95, (f"{st['label']}: its checkpoint fills only {st['coverage']:.1%} of the model's "
                                        "parameters, so backbone_type or model_name doesn't match the checkpoint")
    processor = Sam3Processor(model, device=device, confidence_threshold=CONFIDENCE)
    return model, processor, ckpt


def segment(processor, image, grid=1, precision="fp16", device=DEVICE):
    """Text prompt 'building' on an RGB uint8 image -> 16-bit instance-id map (0 = background), as in E3."""
    h, w = image.shape[:2]
    ids = np.zeros((h, w), np.uint16)
    next_id = 1
    ctx = (lambda: torch.autocast("cuda", dtype=torch.float16)) if (device == "cuda" and precision == "fp16") else nullcontext
    with torch.no_grad():
        for gy in range(grid):
            for gx in range(grid):
                y0, y1, x0, x1 = gy * h // grid, (gy + 1) * h // grid, gx * w // grid, (gx + 1) * w // grid
                with ctx():
                    state = processor.set_image(Image.fromarray(image[y0:y1, x0:x1]))
                    out = processor.set_text_prompt(prompt=PROMPT, state=state)
                masks = out["masks"][:, 0].cpu().numpy() if out["masks"].ndim == 4 else out["masks"].cpu().numpy()
                scores = out["scores"].float().cpu().numpy()
                tile = ids[y0:y1, x0:x1]
                for k in np.argsort(scores):
                    if masks[k].sum() >= MIN_INSTANCE_PX:
                        tile[masks[k] > 0] = next_id
                        next_id += 1
    return ids

def n_instances(ids):
    return int(len(np.unique(ids)) - 1)

probe_pre = cv2.cvtColor(cv2.imread(PROBE_PAIR["pre_path"]), cv2.COLOR_BGR2RGB)
cpu_pairs = [p for p in val_pairs if p["clean"]][:LATENCY_CPU_PAIRS]
cpu_images = [cv2.cvtColor(cv2.imread(p["pre_path"]), cv2.COLOR_BGR2RGB) for p in cpu_pairs]

smoke_rows = []
CHOSEN_PRECISION = {}
for st in STUDENTS:
    t0 = time.time()
    model, processor, ckpt = load_student(st, device=DEVICE)
    load_s = time.time() - t0
    params_m = round(sum(p.numel() for p in model.parameters()) / 1e6, 2)
    st["params_m"] = params_m
    print(f"\n✓ {st['label']} loaded in {load_s:.1f} s: {params_m} M parameters (README: {st['readme_params_m']} M), "
          f"{(st['size_bytes'] or 0)/1e6:.0f} MB file; the checkpoint fills {st['coverage']:.1%} of the model")

    ref_union = None
    for prec in ("fp32", "fp16"):
        row = {"student": st["label"], "rfilename": st["rfilename"], "params_m": params_m,
               "file_mb": round((st["size_bytes"] or 0) / 1e6, 1), "ckpt_coverage": st["coverage"], "precision": prec,
               "t4_ms_grid1": np.nan, "t4_ms_grid2": np.nan, "peak_gpu_gb": np.nan,
               "probe_instances_grid1": np.nan, "iou_vs_fp32": np.nan, "cpu_ms_grid1": np.nan,
               "cpu_ms_grid2": np.nan, "error": ""}
        try:
            torch.cuda.empty_cache()
            torch.cuda.reset_peak_memory_stats()
            segment(processor, probe_pre, grid=1, precision=prec, device=DEVICE)  # warm-up
            t_g1, t_g2 = [], []
            for _ in range(3):
                torch.cuda.synchronize()
                t = time.perf_counter()
                ids1 = segment(processor, probe_pre, grid=1, precision=prec, device=DEVICE)
                torch.cuda.synchronize()
                t_g1.append((time.perf_counter() - t) * 1000)
                t = time.perf_counter()
                ids2 = segment(processor, probe_pre, grid=2, precision=prec, device=DEVICE)
                torch.cuda.synchronize()
                t_g2.append((time.perf_counter() - t) * 1000)
            union = ids1 > 0
            if ref_union is None:
                ref_union = union
            iou_fp32 = float((union & ref_union).sum() / max((union | ref_union).sum(), 1)) if (union | ref_union).sum() else 1.0
            row.update(
                t4_ms_grid1=round(float(np.median(t_g1)), 1),
                t4_ms_grid2=round(float(np.median(t_g2)), 1),
                peak_gpu_gb=round(torch.cuda.max_memory_allocated() / 1e9, 2),
                probe_instances_grid1=n_instances(ids1),
                iou_vs_fp32=round(iou_fp32, 4),
            )
        except Exception as e:
            row["error"] = f"{type(e).__name__}: {str(e)[:140]}"
        smoke_rows.append(row)

    del model, processor
    gc.collect()
    torch.cuda.empty_cache()

    # CPU latency benchmark (4 threads, fp32) on cpu_images
    torch.set_num_threads(CPU_THREADS)
    model_cpu, proc_cpu, _ = load_student(st, device="cpu")
    segment(proc_cpu, cpu_images[0], grid=1, precision="fp32", device="cpu")  # warm-up
    cpu_g1 = []
    for img in tqdm(cpu_images, desc=f"{st['label']} CPU grid1"):
        t = time.perf_counter()
        segment(proc_cpu, img, grid=1, precision="fp32", device="cpu")
        cpu_g1.append((time.perf_counter() - t) * 1000)
    cpu_med_g1 = round(float(np.median(cpu_g1)), 1)
    # grid2 on CPU is 4x grid1; measure on first 5 pairs to bound CPU time
    cpu_g2 = []
    for img in cpu_images[:min(5, len(cpu_images))]:
        t = time.perf_counter()
        segment(proc_cpu, img, grid=2, precision="fp32", device="cpu")
        cpu_g2.append((time.perf_counter() - t) * 1000)
    cpu_med_g2 = round(float(np.median(cpu_g2)), 1)
    for r in smoke_rows:
        if r["student"] == st["label"]:
            r["cpu_ms_grid1"] = cpu_med_g1
            r["cpu_ms_grid2"] = cpu_med_g2
    del model_cpu, proc_cpu
    gc.collect()

    sub = pd.DataFrame([r for r in smoke_rows if r["student"] == st["label"] and r["error"] == "" and r["iou_vs_fp32"] >= 0.95])
    assert not sub.empty, f"{st['label']} failed in both precisions"
    CHOSEN_PRECISION[st["label"]] = sub.sort_values("t4_ms_grid2")["precision"].iloc[0]

# Add E3's SAM 3 reference row for direct speed/size/memory comparison
smoke_rows.append({
    "student": "SAM 3 (E3 ref)", "rfilename": "facebook/sam3 (sam3.pt)", "params_m": 841.0,
    "file_mb": 3450.0, "precision": "fp16", "t4_ms_grid1": 540.0, "t4_ms_grid2": 2110.0,
    "peak_gpu_gb": 4.07, "probe_instances_grid1": 11, "iou_vs_fp32": 0.9998,
    "cpu_ms_grid1": np.nan, "cpu_ms_grid2": np.nan, "error": "",
})
smoke_df = pd.DataFrame(smoke_rows)
smoke_df.to_csv(OUT_DIR / "smoke_and_latency.csv", index=False)
print("\n✓ Part 1 smoke test & latency table:")
print(smoke_df.drop(columns=["rfilename", "error"]).to_string(index=False))

In [ ]:
# ── Cell 6: Part 2: building finding on the validation before-photos (pre_grid1 and pre_grid2) ──
def building_blobs(mask):
    return int(cv2.connectedComponents((mask > 0).astype(np.uint8), connectivity=8)[0] - 1)

def score_row(found, gt, instances):
    return {
        "inter": int((found & (gt > 0)).sum()),
        "union": int((found | (gt > 0)).sum()),
        "instances": int(instances),
        "gt_blobs": building_blobs(gt),
        **{f"gt_{c}_px": int((gt == i).sum()) for i, c in CLASSES},
        **{f"found_{c}_px": int((found & (gt == i)).sum()) for i, c in CLASSES},
    }

# Self-check on a synthetic mask
_gt = np.array([[0, 1], [2, 3]], dtype=np.uint8)
_sc = score_row(_gt > 0, _gt, 1)
assert _sc["inter"] == 3 and _sc["union"] == 3 and _sc["found_destroyed_px"] == 1

rows = []
for st in STUDENTS:
    label = st["label"]
    prec = CHOSEN_PRECISION[label]
    model, processor, _ = load_student(st, device=DEVICE)
    for kind, grid in RUNS:
        run_name = f"{label}_{kind}_grid{grid}"
        for pair in tqdm(val_pairs, desc=run_name):
            image = cv2.imread(pair["pre_path"])
            gt = cv2.imread(pair["mask_path"], cv2.IMREAD_GRAYSCALE)
            if image is None or gt is None:
                continue
            image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
            if gt.shape != image.shape[:2]:
                gt = cv2.resize(gt, (image.shape[1], image.shape[0]), interpolation=cv2.INTER_NEAREST)
            torch.cuda.synchronize()
            t = time.perf_counter()
            ids = segment(processor, image, grid=grid, precision=prec, device=DEVICE)
            torch.cuda.synchronize()
            sec = time.perf_counter() - t
            rows.append({
                "model": label, "run": f"{kind}_grid{grid}", "full_run": run_name,
                "image": pair["image"], "event": pair["event"], "source": pair["source"],
                "clean": pair["clean"], "sec": round(sec, 3),
                **score_row(ids > 0, gt, n_instances(ids)),
            })
    del model, processor
    gc.collect()
    torch.cuda.empty_cache()

per_image = pd.DataFrame(rows)
per_image.to_csv(OUT_DIR / "l4_per_image.csv", index=False)
print(f"✓ Scored {len(per_image)} student-image runs: {per_image.groupby('full_run').size().to_dict()}")

In [ ]:
# ── Cell 7: Summary on all 645 and clean 605, comparison with SAM 3 and R9, verdict and README ──
# Combine EfficientSAM3 runs with E3's SAM 3 reference rows on the exact same validation pairs
val_names = {p["image"] for p in val_pairs}
e3_sub = E3_DF[E3_DF["image"].isin(val_names) & E3_DF["run"].isin(["pre_grid1", "pre_grid2"])].copy()
e3_sub["model"] = "SAM 3 (E3)"
e3_sub["full_run"] = "SAM 3_" + e3_sub["run"]
combined = pd.concat([per_image, e3_sub[per_image.columns]], ignore_index=True)

summary_rows = []
for (mdl, run), d in combined.groupby(["model", "run"], sort=False):
    for split_name, split_df in (("all_645", d), ("clean_605", d[d["clean"]])):
        for src, g in [("all", split_df), *split_df.groupby("source")]:
            if len(g):
                summary_rows.append({"model": mdl, "run": run, "split": split_name, "source": src, **summarize_df(g)})

# Also add R9 TTA reference rows from E3 / E5
if not SMOKE_TEST:
    summary_rows.extend([
        {"model": "R9_tta (E1/E5)", "run": "post_tta", "split": "all_645", "source": "all", "images": 645,
         "building_iou": 0.5905, "count_ratio": 0.615, "count_mae": 16.3, "recall_intact": 0.8469,
         "recall_damaged": 0.7851, "recall_destroyed": 0.7090, "sec_per_image": np.nan},
        {"model": "R9_tta (E1/E5)", "run": "post_tta", "split": "clean_605", "source": "all", "images": 605,
         "building_iou": 0.5905, "count_ratio": np.nan, "count_mae": np.nan, "recall_intact": np.nan,
         "recall_damaged": np.nan, "recall_destroyed": np.nan, "sec_per_image": np.nan},
    ])

summary = pd.DataFrame(summary_rows)
summary.to_csv(OUT_DIR / "L4_summary.csv", index=False)

clean_all = summary[(summary["split"] == "clean_605") & (summary["source"] == "all")].copy()
print("✓ Clean validation pairs (source = all):")
print(clean_all.to_string(index=False))

# Evaluate the rule fixed before the run (on clean_605, pre_grid2)
R9_BAR = 0.5905
SAM3_CLEAN = 0.6302
MIN_BAR = max(R9_BAR, round(SAM3_CLEAN - 0.05, 4))
g2_clean = clean_all[clean_all["run"] == "pre_grid2"].set_index("model")
passing = []
# The rule's second condition: faster and less GPU memory than full SAM 3 (2.11 s/image and 4.07 GB on the T4).
# Memory is the Part 1 peak at the precision Part 2 used.
peak_gb = smoke_df.set_index(["student", "precision"])["peak_gpu_gb"]
for st in STUDENTS:
    lbl = st["label"]
    if lbl in g2_clean.index:
        biou = float(g2_clean.loc[lbl, "building_iou"])
        sec = float(g2_clean.loc[lbl, "sec_per_image"])
        mem = float(peak_gb.get((lbl, CHOSEN_PRECISION[lbl]), np.nan))
        if biou >= MIN_BAR and sec < 2.11 and mem < 4.07:
            passing.append((lbl, biou, sec))

if passing:
    best_lbl, best_iou, best_sec = max(passing, key=lambda x: x[1])
    verdict = (f"{best_lbl} PASSES L4's rule on the 605 clean pairs: pre_grid2 Building IoU {best_iou:.4f} "
               f"(>= {MIN_BAR:.4f}, vs SAM 3's {SAM3_CLEAN} and R9's {R9_BAR}) at {best_sec:.2f} s/image.")
else:
    best_st = max(STUDENTS, key=lambda s: float(g2_clean.loc[s["label"], "building_iou"]) if s["label"] in g2_clean.index else -1)
    best_lbl = best_st["label"]
    best_iou = float(g2_clean.loc[best_lbl, "building_iou"]) if best_lbl in g2_clean.index else float("nan")
    best_sec = float(g2_clean.loc[best_lbl, "sec_per_image"]) if best_lbl in g2_clean.index else float("nan")
    verdict = (f"No EfficientSAM3 student passes L4's rule (bar >= {MIN_BAR:.4f} on the 605 clean pairs, faster than "
               f"2.11 s/image, under 4.07 GB): the best is {best_lbl} at pre_grid2 Building IoU {best_iou:.4f} "
               f"({best_sec:.2f} s/image), vs full SAM 3's {SAM3_CLEAN} and R9's {R9_BAR}. Full SAM 3 stays the "
               "base-station finder and the 8-bit U-Net (L1b/L1c, 33 MB) stays the field model.")
print(f"\nVerdict: {verdict}")

def as_md(df):
    try:
        return df.to_markdown(index=False)
    except ImportError:
        return "```\n" + df.to_string(index=False) + "\n```"

config = {
    "task": "L4", "smoke_test": SMOKE_TEST, "efficientsam3_commit": ESAM3_COMMIT,
    "hf_repo": f"{HF_REPO}@{HF_SHA[:8]}", "students": STUDENTS, "chosen_precision": CHOSEN_PRECISION,
    "prompt": PROMPT, "confidence": CONFIDENCE, "runs": RUNS, "min_instance_px": MIN_INSTANCE_PX,
    "val_pairs": len(val_pairs), "clean_pairs": int(sum(p["clean"] for p in val_pairs)),
    "rule_bar_building_iou": MIN_BAR, "verdict": verdict,
}
(OUT_DIR / "config.json").write_text(json.dumps(config, indent=1), encoding="utf-8")
(OUT_DIR / "README.md").write_text("\n".join([
    f"# L4 results: EfficientSAM3 vs SAM 3 building finding{' (SMOKE TEST)' if SMOKE_TEST else ''}", "",
    f"Repo `SimonZeng7108/efficientsam3@{ESAM3_COMMIT[:7]}`, weights `{HF_REPO}@{HF_SHA[:8]}`, prompt `'{PROMPT}'`, "
    f"confidence {CONFIDENCE}, {len(val_pairs)} validation pairs ({sum(p['clean'] for p in val_pairs)} clean).", "",
    "## Part 1: model size, T4 precision check, and T4 vs CPU latency", "",
    as_md(smoke_df.drop(columns=["rfilename", "error"])), "",
    "## Part 2: building finding on the 605 clean validation pairs (source = all)", "",
    as_md(clean_all), "",
    "## Full breakdown (all 645 and clean 605, by source)", "",
    as_md(summary), "",
    "## Verdict", "", verdict, "",
    "## Files", "",
    "- `smoke_and_latency.csv`, `l4_per_image.csv`, `L4_summary.csv`, `config.json`, `README.md`",
]) + "\n", encoding="utf-8")
print(f"✓ Wrote smoke_and_latency.csv, l4_per_image.csv, L4_summary.csv, config.json and README.md to {OUT_DIR}")